# Bab 5: Klasifikasi Linear

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab05/data_sms.py": "\"\"\"Korpus SMS sintetis berbahasa Indonesia (ilustrasi, bukan pesan asli).\n\nPesan dibangkitkan dari templat yang meniru pola umum penipuan (hadiah,\n\"mama minta pulsa\", pinjol, kurir/APK, perubahan tarif bank) dan pesan\nsehari-hari. Semua nomor, tautan, dan nama adalah rekaan.\n\"\"\"\nimport numpy as np\n\nPENIPUAN = {\n    \"hadiah\": [\"SELAMAT! No anda terpilih mendapat hadiah {hadiah} dari {usaha}. Info klik {url} PIN: {pin}\",\n               \"Selamat anda pemenang undian {usaha} ke-{pin}, hadiah {hadiah}. Hub {hp}\",\n               \"PEMENANG {hadiah}!! Segera klaim di {url} sebelum hangus\"],\n    \"keluarga\": [\"Ma ini nomor baru papa, tolong kirim pulsa {nominal} ke nomor ini dulu ya penting\",\n                 \"Bu ini aku, hp ku rusak. tolong transfer {rp} ke rek {bank} {norek} a/n {nama}\",\n                 \"Pak, saya {nama} temen anak bapak, dia kecelakaan, butuh {rp} sekarang ke {norek}\",\n                 \"TF {rp} ke {norek} {bank} skrg, konfirm {hp}\",\n                 \"Kak ini aku, bisa pinjam {rp} dulu? nanti diganti hari {hari}\"],\n    \"pinjol\": [\"Pinjaman dana cepat cair {rp} tanpa jaminan, bunga 0%. Daftar via WA {hp}\",\n               \"BUTUH DANA? cair 5 menit {rp}, cukup KTP. klik {url}\",\n               \"Limit pinjaman anda {rp} sudah aktif, ambil sekarang di {url}\"],\n    \"kurir\": [\"Paket anda tertahan. Lihat foto resi di aplikasi berikut: {apk}\",\n              \"Kurir {kurir}: paket tidak dapat dikirim, konfirmasi alamat di {url}\",\n              \"Ini foto paketnya kak, tolong dicek {apk}\"],\n    \"bank\": [\"Nasabah {bank} Yth, tarif transfer berubah jadi Rp150.000/bln. Jika tidak setuju isi {url}\",\n             \"Akun {bank} anda DIBLOKIR. Verifikasi data di {url} dalam 1x24 jam\",\n             \"Poin {bank} anda {pin} akan hangus, tukarkan di {url}\"],\n}\nNORMAL = [\"{sapa}, nanti jam {jam} jadi ketemu di {tempat}?\",\n          \"Bu, besok rapat prodi jam {jam} di ruang {ruang}\",\n          \"Mas, uang kos bulan ini sudah saya transfer ya {rp}\",\n          \"Pesanan kamu sudah sampai, ditaruh di depan pintu\",\n          \"Kode OTP anda {pin}. JANGAN berikan kode ini kepada siapapun\",\n          \"Paket {kurir} dengan resi {resi} sedang diantar kurir\",\n          \"Jangan lupa beli {barang} pas pulang ya\",\n          \"Assalamualaikum, kajian ba'da maghrib di masjid {masjid} malam ini\",\n          \"Tagihan listrik bulan ini {rp}, sudah dibayar belum?\",\n          \"Selamat ulang tahun {nama}! semoga sehat selalu\",\n          \"{sapa} tugas kelompok dikumpul hari {hari} ya, jangan telat\",\n          \"Ok siap, nanti saya kabari lagi\",\n          \"Pak, mohon izin hari ini tidak masuk karena sakit\",\n          \"Transaksi {bank} sebesar {rp} berhasil pada {jam}\",\n          \"Mak, aku pulang hari {hari} naik travel jam {jam}\",\n          \"Promo {usaha}: diskon 50% hari ini, cek aplikasinya\",\n          \"Pinjam dulu {rp} ya, nanti aku ganti hari {hari}\",\n          \"No rek aku {bank} {norek} a/n {nama}, makasih ya\"]\n\nSELIPAN = [\"ya\", \"deh\", \"kak\", \"sekarang\", \"info\", \"segera\", \"makasih\", \"penting\",\n           \"cek\", \"tolong\", \"dulu\", \"bisa\", \"nanti\", \"hari\", \"ini\", \"aja\", \"gan\", \"bu\"]\n\nISI = dict(hadiah=[\"mobil Avanza\", \"uang tunai 100jt\", \"motor NMAX\", \"emas 10gr\"],\n           usaha=[\"PT Sinar Jaya\", \"Shopee\", \"Bank Rakyat\", \"Telkomsel\"],\n           url=[\"bit.ly/hdh-{n}\", \"klaim-{n}.xyz\", \"s.id/promo{n}\", \"cek-resi{n}.top\"],\n           apk=[\"resi_paket.apk\", \"foto-paket{n}.apk\", \"lihat_resi.apk\"],\n           bank=[\"BRI\", \"BNI\", \"Mandiri\", \"BCA\", \"BSI\"], kurir=[\"JNE\", \"J&T\", \"SiCepat\", \"POS\"],\n           nama=[\"Rina\", \"Andi\", \"Siti\", \"Budi\", \"Dewi\", \"Fajar\"],\n           sapa=[\"Bro\", \"Kak\", \"Dek\", \"Mbak\", \"Bang\"], tempat=[\"kampus\", \"kantin\", \"masjid\", \"mall\"],\n           ruang=[\"301\", \"rapat\", \"dosen\"], barang=[\"telur\", \"beras\", \"gas\", \"galon\"],\n           masjid=[\"Al-Ikhlas\", \"Raya\", \"Nurul Iman\"], hari=[\"Senin\", \"Rabu\", \"Jumat\", \"Sabtu\"])\n\ndef _isi(templat, rng):\n    nilai = {k: rng.choice(v).replace(\"{n}\", str(rng.integers(10, 999))) for k, v in ISI.items()}\n    nilai.update(pin=str(rng.integers(1000, 99999)), resi=f\"JP{rng.integers(10**9, 10**10)}\",\n                 hp=f\"08{rng.integers(10**9, 10**10)}\", norek=str(rng.integers(10**9, 10**10)),\n                 jam=f\"{rng.integers(7, 22)}.{rng.choice(['00', '30'])}\",\n                 rp=\"Rp\" + f\"{int(rng.integers(1, 50)) * 50_000:,}\".replace(\",\", \".\"), nominal=f\"{rng.choice([20, 50, 100])}rb\")\n    kata = templat.format(**nilai).split()\n    if len(kata) > 4 and rng.random() < 0.5:           # pesan nyata tidak rapi:\n        del kata[rng.integers(len(kata))]                # kata hilang,\n    for _ in range(rng.poisson(1.0)):                    # kata tambahan acak\n        kata.insert(rng.integers(len(kata) + 1), rng.choice(SELIPAN))\n    return \" \".join(kata)\n\ndef buat_korpus(n_normal=1500, n_penipuan=500, seed=0):\n    \"\"\"Kembalikan (teks, label 0/1, jenis penipuan atau 'normal').\"\"\"\n    rng = np.random.default_rng(seed)\n    teks, label, jenis = [], [], []\n    for _ in range(n_normal):\n        teks.append(_isi(rng.choice(NORMAL), rng)); label.append(0); jenis.append(\"normal\")\n    nama_jenis = list(PENIPUAN)\n    for _ in range(n_penipuan):\n        j = rng.choice(nama_jenis)\n        teks.append(_isi(rng.choice(PENIPUAN[j]), rng)); label.append(1); jenis.append(j)\n    balik = rng.random(len(label)) < 0.02               # 2% label keliru (anotator)\n    label = [1 - l if b else l for l, b in zip(label, balik)]\n    urut = rng.permutation(len(teks))\n    return ([teks[i] for i in urut], np.array(label)[urut], np.array(jenis)[urut])\n\ndef _fitur_dua_tidak_dipakai(teks):\n    \"\"\"Dua fitur sederhana: proporsi angka dan proporsi huruf kapital.\"\"\"\n    angka = np.array([sum(c.isdigit() for c in s) / len(s) for s in teks])\n    kapital = np.array([sum(c.isupper() for c in s) / len(s) for s in teks])\n    return np.column_stack([angka, kapital])\n\ndef normalisasi(s):\n    \"\"\"Huruf kecil, setiap deret angka diganti '0' agar nomor tidak jadi kata unik.\"\"\"\n    import re\n    return re.sub(r\"\\d+\", \"0\", s.lower())\n\ndef kantong_kata(teks, latih):\n    \"\"\"Matriks bag of words biner; kosakata hanya dari data latih (min. 3 pesan).\"\"\"\n    from sklearn.feature_extraction.text import CountVectorizer\n    vek = CountVectorizer(preprocessor=normalisasi, token_pattern=r\"[a-z0-9&]+\",\n                          min_df=3, binary=True)\n    vek.fit([s for s, l in zip(teks, latih) if l])\n    return vek.transform(teks).toarray().astype(float), vek.get_feature_names_out()\n",
"bab05/kuadrat_fisher.py": "import numpy as np\nfrom data_sms import buat_korpus, kantong_kata\n\nteks, t, _ = buat_korpus()\nlatih = np.arange(len(t)) < 1400                    # 1400 latih, 600 uji\nX, kata = kantong_kata(teks, latih)\nprint(f\"kosakata: {len(kata)} kata\")\n\n# Kuadrat terkecil dengan target 0/1  #<1>\nPhi = np.column_stack([np.ones(len(t)), X])\nb = np.linalg.lstsq(Phi[latih], t[latih], rcond=None)[0]\ny = Phi[~latih] @ b\nprint(f\"kuadrat terkecil: keluaran uji di [{y.min():.2f}, {y.max():.2f}], \"\n      f\"{np.mean((y < 0) | (y > 1)):.0%} di luar [0, 1]\")\n\n# Proyeksi ke satu dimensi: selisih rerata vs Fisher  #<2>\nX0, X1 = X[latih & (t == 0)], X[latih & (t == 1)]\nm0, m1 = X0.mean(0), X1.mean(0)\nSw = (X0 - m0).T @ (X0 - m0) + (X1 - m1).T @ (X1 - m1)\narah = {\"selisih rerata\": m1 - m0,\n        \"Fisher\": np.linalg.solve(Sw + 1e-1 * np.eye(len(kata)), m1 - m0)}  #<3>\nfor nama, w in arah.items():\n    z0, z1 = X0 @ w, X1 @ w\n    J = (z1.mean() - z0.mean())**2 / (z0.var() + z1.var())   # kriteria Fisher\n    zu = X[~latih] @ w\n    ambang = 0.5 * (z0.mean() + z1.mean())\n    akurasi = np.mean((zu > ambang) == t[~latih])\n    print(f\"{nama:15s}: J = {J:6.2f} | akurasi uji = {akurasi:.3f}\")\n",
"bab05/laplace_logistik.py": "import numpy as np\nfrom data_sms import buat_korpus, kantong_kata\n\nteks, t, _ = buat_korpus()\nlatih = np.arange(len(t)) < 1400\nbaru = [\"Kak ini foto paketnya, tolong cek dulu\",\n        \"Selamat ulang tahun kak, semoga sehat\",\n        \"Lowongan kerja paruh waktu, gaji harian 300rb, minat?\",   # jenis baru  #<1>\n        \"Bu, transfer dulu ya ke rek BRI, penting\"]\nX, _ = kantong_kata(list(teks) + baru, np.r_[latih, np.zeros(len(baru), bool)])\nPhi = np.column_stack([np.ones(len(X)), X])[-len(baru):]\n\nw = np.load(\"w_map.npy\")                            # mode posterior (IRLS)\nS = np.linalg.inv(np.load(\"H_map.npy\"))             # kovarians Laplace = H^-1  #<2>\nsigmoid = lambda a: 1 / (1 + np.exp(-a))\n\nmu_a = Phi @ w                                      # rerata aktivasi\nvar_a = np.sum(Phi @ S * Phi, axis=1)               # varians aktivasi\nkappa = 1 / np.sqrt(1 + np.pi * var_a / 8)          # aproksimasi probit  #<3>\nrng = np.random.default_rng(0)                      # pembanding Monte Carlo\nW = rng.multivariate_normal(w, S, size=20_000)\np_mc = sigmoid(Phi @ W.T).mean(axis=1)\n\nprint(\"pesan  kata dikenal   p_MAP   p_Bayes   (Monte Carlo)   sd aktivasi\")\nfor i in range(len(baru)):\n    print(f\"{i+1:5d} {int(X[-len(baru)+i].sum()):13d} {sigmoid(mu_a[i]):7.3f} \"\n          f\"{sigmoid(kappa[i]*mu_a[i]):9.3f} {p_mc[i]:12.3f} {np.sqrt(var_a[i]):13.2f}\")\n",
"bab05/logistik_irls.py": "import numpy as np\nfrom data_sms import buat_korpus, kantong_kata\n\nteks, t, _ = buat_korpus()\nlatih = np.arange(len(t)) < 1400\nX, kata = kantong_kata(teks, latih)\nPhi = np.column_stack([np.ones(len(t)), X])         # tambah konstanta\nP, tl = Phi[latih], t[latih]\nsigmoid = lambda a: 1 / (1 + np.exp(-a))\n\nalpha = 1.0                                         # presisi prior Gaussian\nw = np.zeros(P.shape[1])\nfor it in range(1, 30):                             # IRLS = metode Newton  #<1>\n    y = sigmoid(P @ w)\n    g = P.T @ (y - tl) + alpha * w                  # gradien  #<2>\n    H = P.T @ (P * (y * (1 - y))[:, None]) + alpha * np.eye(len(w))  # Hessian\n    langkah = np.linalg.solve(H, g)\n    w -= langkah\n    if np.abs(langkah).max() < 1e-8:\n        break\nprint(f\"IRLS konvergen dalam {it} iterasi\")\n\np = sigmoid(Phi[~latih] @ w)\ntu = t[~latih]\nlog_loss = np.mean(np.logaddexp(0, -(2 * tu - 1) * (Phi[~latih] @ w)))   #<3>\nprint(f\"akurasi uji: {np.mean((p > 0.5) == tu):.3f} | log-loss uji: {log_loss:.3f}\")\nprint(f\"prediksi sangat yakin (p<0.001 atau p>0.999): {np.mean((p < 1e-3) | (p > 1 - 1e-3)):.0%}\")\nnp.save(\"w_map.npy\", w); np.save(\"H_map.npy\", H)   # untuk aproksimasi Laplace\n",
"bab05/naive_bayes_sms.py": "import numpy as np\nfrom data_sms import buat_korpus, kantong_kata\n\nteks, t, _ = buat_korpus()\nlatih = np.arange(len(t)) < 1400\nX, kata = kantong_kata(teks, latih)\nXl, tl = X[latih], t[latih]\n\n# Bernoulli naive Bayes: p(kata muncul | kelas), dengan prior Beta(1, 1)  #<1>\npi = np.array([np.mean(tl == 0), np.mean(tl == 1)])\nmu = np.array([(Xl[tl == k].sum(0) + 1) / ((tl == k).sum() + 2) for k in (0, 1)])\n\ndef log_posterior_odds(X):                          #<2>\n    lp = [np.log(pi[k]) + X @ np.log(mu[k]) + (1 - X) @ np.log(1 - mu[k]) for k in (0, 1)]\n    return lp[1] - lp[0]                            # a = ln p(C1|x) - ln p(C0|x)\n\na = log_posterior_odds(X[~latih])\np = 1 / (1 + np.exp(-a))                            # sigmoid logistik  #<3>\nlog_loss = np.mean(np.logaddexp(0, -(2 * t[~latih] - 1) * a))   # stabil numerik\nprint(f\"akurasi uji: {np.mean((p > 0.5) == t[~latih]):.3f} | log-loss uji: {log_loss:.3f}\")\nprint(f\"prediksi sangat yakin (p<0.001 atau p>0.999): {np.mean((p < 1e-3) | (p > 1 - 1e-3)):.0%}\")\n\nbobot = np.log(mu[1] / mu[0])                       # sumbangan tiap kata  #<4>\nurut = np.argsort(bobot)\nprint(\"paling 'penipuan':\", \", \".join(kata[urut[-8:][::-1]]))\nprint(\"paling 'normal'  :\", \", \".join(kata[urut[:8]]))\n",
"bab05/softmax_jenis.py": "import numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom data_sms import buat_korpus, kantong_kata\n\nteks, _, jenis = buat_korpus(n_normal=1500, n_penipuan=1500, seed=1)\nbaru = [\"Selamat paket anda pemenang, klik foto resi\",          # campuran  #<1>\n        \"Kak pinjam dana dulu ya, nanti aku transfer balik\",\n        \"Besok rapat jam 9, jangan lupa bawa laptop\"]\nsemua = list(teks) + baru\nlatih = np.r_[np.ones(len(teks), bool), np.zeros(len(baru), bool)]\nX, _ = kantong_kata(semua, latih)\n\nmodel = LogisticRegression(C=1.0, max_iter=2000)    # softmax dengan prior L2  #<2>\nmodel.fit(X[latih], jenis)\nP = model.predict_proba(X[~latih])                  # satu baris = 6 peluang  #<3>\nprint(\" \" * 9 + \"\".join(f\"{k:>9s}\" for k in model.classes_))\nfor i, p in enumerate(P, start=1):\n    print(f\"pesan {i}: \" + \"\".join(f\"{v:9.2f}\" for v in p))\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab05')
print('folder kerja:', os.getcwd())

## Listing 5.1: Kuadrat terkecil dan diskriminan Fisher pada bag of words SMS

Berkas: `kode/bab05/kuadrat_fisher.py`

In [ ]:
import numpy as np
from data_sms import buat_korpus, kantong_kata

teks, t, _ = buat_korpus()
latih = np.arange(len(t)) < 1400                    # 1400 latih, 600 uji
X, kata = kantong_kata(teks, latih)
print(f"kosakata: {len(kata)} kata")

# Kuadrat terkecil dengan target 0/1  # (1)
Phi = np.column_stack([np.ones(len(t)), X])
b = np.linalg.lstsq(Phi[latih], t[latih], rcond=None)[0]
y = Phi[~latih] @ b
print(f"kuadrat terkecil: keluaran uji di [{y.min():.2f}, {y.max():.2f}], "
      f"{np.mean((y < 0) | (y > 1)):.0%} di luar [0, 1]")

# Proyeksi ke satu dimensi: selisih rerata vs Fisher  # (2)
X0, X1 = X[latih & (t == 0)], X[latih & (t == 1)]
m0, m1 = X0.mean(0), X1.mean(0)
Sw = (X0 - m0).T @ (X0 - m0) + (X1 - m1).T @ (X1 - m1)
arah = {"selisih rerata": m1 - m0,
        "Fisher": np.linalg.solve(Sw + 1e-1 * np.eye(len(kata)), m1 - m0)}  # (3)
for nama, w in arah.items():
    z0, z1 = X0 @ w, X1 @ w
    J = (z1.mean() - z0.mean())**2 / (z0.var() + z1.var())   # kriteria Fisher
    zu = X[~latih] @ w
    ambang = 0.5 * (z0.mean() + z1.mean())
    akurasi = np.mean((zu > ambang) == t[~latih])
    print(f"{nama:15s}: J = {J:6.2f} | akurasi uji = {akurasi:.3f}")

## Listing 5.2: Naive Bayes Bernoulli dari nol untuk SMS penipuan

Berkas: `kode/bab05/naive_bayes_sms.py`

In [ ]:
import numpy as np
from data_sms import buat_korpus, kantong_kata

teks, t, _ = buat_korpus()
latih = np.arange(len(t)) < 1400
X, kata = kantong_kata(teks, latih)
Xl, tl = X[latih], t[latih]

# Bernoulli naive Bayes: p(kata muncul | kelas), dengan prior Beta(1, 1)  # (1)
pi = np.array([np.mean(tl == 0), np.mean(tl == 1)])
mu = np.array([(Xl[tl == k].sum(0) + 1) / ((tl == k).sum() + 2) for k in (0, 1)])

def log_posterior_odds(X):                          # (2)
    lp = [np.log(pi[k]) + X @ np.log(mu[k]) + (1 - X) @ np.log(1 - mu[k]) for k in (0, 1)]
    return lp[1] - lp[0]                            # a = ln p(C1|x) - ln p(C0|x)

a = log_posterior_odds(X[~latih])
p = 1 / (1 + np.exp(-a))                            # sigmoid logistik  # (3)
log_loss = np.mean(np.logaddexp(0, -(2 * t[~latih] - 1) * a))   # stabil numerik
print(f"akurasi uji: {np.mean((p > 0.5) == t[~latih]):.3f} | log-loss uji: {log_loss:.3f}")
print(f"prediksi sangat yakin (p<0.001 atau p>0.999): {np.mean((p < 1e-3) | (p > 1 - 1e-3)):.0%}")

bobot = np.log(mu[1] / mu[0])                       # sumbangan tiap kata  # (4)
urut = np.argsort(bobot)
print("paling 'penipuan':", ", ".join(kata[urut[-8:][::-1]]))
print("paling 'normal'  :", ", ".join(kata[urut[:8]]))

## Listing 5.3: Regresi logistik dengan IRLS dan prior Gaussian, dari nol

Berkas: `kode/bab05/logistik_irls.py`

In [ ]:
import numpy as np
from data_sms import buat_korpus, kantong_kata

teks, t, _ = buat_korpus()
latih = np.arange(len(t)) < 1400
X, kata = kantong_kata(teks, latih)
Phi = np.column_stack([np.ones(len(t)), X])         # tambah konstanta
P, tl = Phi[latih], t[latih]
sigmoid = lambda a: 1 / (1 + np.exp(-a))

alpha = 1.0                                         # presisi prior Gaussian
w = np.zeros(P.shape[1])
for it in range(1, 30):                             # IRLS = metode Newton  # (1)
    y = sigmoid(P @ w)
    g = P.T @ (y - tl) + alpha * w                  # gradien  # (2)
    H = P.T @ (P * (y * (1 - y))[:, None]) + alpha * np.eye(len(w))  # Hessian
    langkah = np.linalg.solve(H, g)
    w -= langkah
    if np.abs(langkah).max() < 1e-8:
        break
print(f"IRLS konvergen dalam {it} iterasi")

p = sigmoid(Phi[~latih] @ w)
tu = t[~latih]
log_loss = np.mean(np.logaddexp(0, -(2 * tu - 1) * (Phi[~latih] @ w)))   # (3)
print(f"akurasi uji: {np.mean((p > 0.5) == tu):.3f} | log-loss uji: {log_loss:.3f}")
print(f"prediksi sangat yakin (p<0.001 atau p>0.999): {np.mean((p < 1e-3) | (p > 1 - 1e-3)):.0%}")
np.save("w_map.npy", w); np.save("H_map.npy", H)   # untuk aproksimasi Laplace

## Listing 5.4: Regresi softmax untuk jenis penipuan

Berkas: `kode/bab05/softmax_jenis.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from data_sms import buat_korpus, kantong_kata

teks, _, jenis = buat_korpus(n_normal=1500, n_penipuan=1500, seed=1)
baru = ["Selamat paket anda pemenang, klik foto resi",          # campuran  # (1)
        "Kak pinjam dana dulu ya, nanti aku transfer balik",
        "Besok rapat jam 9, jangan lupa bawa laptop"]
semua = list(teks) + baru
latih = np.r_[np.ones(len(teks), bool), np.zeros(len(baru), bool)]
X, _ = kantong_kata(semua, latih)

model = LogisticRegression(C=1.0, max_iter=2000)    # softmax dengan prior L2  # (2)
model.fit(X[latih], jenis)
P = model.predict_proba(X[~latih])                  # satu baris = 6 peluang  # (3)
print(" " * 9 + "".join(f"{k:>9s}" for k in model.classes_))
for i, p in enumerate(P, start=1):
    print(f"pesan {i}: " + "".join(f"{v:9.2f}" for v in p))

## Listing 5.5: Regresi logistik Bayesian dengan aproksimasi Laplace

Berkas: `kode/bab05/laplace_logistik.py`

In [ ]:
import numpy as np
from data_sms import buat_korpus, kantong_kata

teks, t, _ = buat_korpus()
latih = np.arange(len(t)) < 1400
baru = ["Kak ini foto paketnya, tolong cek dulu",
        "Selamat ulang tahun kak, semoga sehat",
        "Lowongan kerja paruh waktu, gaji harian 300rb, minat?",   # jenis baru  # (1)
        "Bu, transfer dulu ya ke rek BRI, penting"]
X, _ = kantong_kata(list(teks) + baru, np.r_[latih, np.zeros(len(baru), bool)])
Phi = np.column_stack([np.ones(len(X)), X])[-len(baru):]

w = np.load("w_map.npy")                            # mode posterior (IRLS)
S = np.linalg.inv(np.load("H_map.npy"))             # kovarians Laplace = H^-1  # (2)
sigmoid = lambda a: 1 / (1 + np.exp(-a))

mu_a = Phi @ w                                      # rerata aktivasi
var_a = np.sum(Phi @ S * Phi, axis=1)               # varians aktivasi
kappa = 1 / np.sqrt(1 + np.pi * var_a / 8)          # aproksimasi probit  # (3)
rng = np.random.default_rng(0)                      # pembanding Monte Carlo
W = rng.multivariate_normal(w, S, size=20_000)
p_mc = sigmoid(Phi @ W.T).mean(axis=1)

print("pesan  kata dikenal   p_MAP   p_Bayes   (Monte Carlo)   sd aktivasi")
for i in range(len(baru)):
    print(f"{i+1:5d} {int(X[-len(baru)+i].sum()):13d} {sigmoid(mu_a[i]):7.3f} "
          f"{sigmoid(kappa[i]*mu_a[i]):9.3f} {p_mc[i]:12.3f} {np.sqrt(var_a[i]):13.2f}")